# Retail KPI Dashboard
**Dataset:** [Superstore Sales Dataset – Kaggle](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final)

> **Business Question:** How is the business performing across regions and categories — and where are the gaps?

---
**Author:** Your Name  
**Tools:** Python · Pandas · Plotly  
**GitHub:** https://github.com/yourusername/retail-dashboard

> 💡 This notebook creates an interactive HTML dashboard saved to `../retail_dashboard.html`  
> Open that file in any browser — no server needed.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings, os
warnings.filterwarnings('ignore')

ACCENT   = '#1a6b4a'
GREEN    = '#2d9966'
LIGHT_G  = '#9FE1CB'
DANGER   = '#c0392b'
WARN     = '#e67e22'
GRAY     = '#7f8c8d'

print('Libraries loaded ✓')

## 2. Load Data

📥 **Download the Superstore dataset from Kaggle:**
1. Go to https://www.kaggle.com/datasets/vivek468/superstore-dataset-final
2. Download `Sample - Superstore.csv`
3. Place it in `../data/`

Or via CLI:
```bash
kaggle datasets download -d vivek468/superstore-dataset-final -p ../data --unzip
```

In [ ]:
data_path = '../data/Sample - Superstore.csv'

if os.path.exists(data_path):
    df = pd.read_csv(data_path, encoding='latin-1')
    print(f'✓ Real dataset: {df.shape[0]:,} rows')
else:
    print('⚠ Generating sample Superstore-style data...')
    np.random.seed(99)
    n = 9994
    regions   = ['West','East','Central','South']
    cats      = ['Technology','Furniture','Office Supplies']
    segments  = ['Consumer','Corporate','Home Office']
    subcats   = {'Technology':['Phones','Copiers','Machines','Accessories'],
                 'Furniture':['Chairs','Tables','Bookcases','Furnishings'],
                 'Office Supplies':['Binders','Paper','Storage','Appliances']}
    ship_modes = ['Standard Class','Second Class','First Class','Same Day']

    cat_col = np.random.choice(cats, n, p=[0.36,0.32,0.32])
    subcat_col = [np.random.choice(subcats[c]) for c in cat_col]
    sales = np.array([{'Technology':1200,'Furniture':800,'Office Supplies':150}[c]
                      * np.random.uniform(0.3,3.5) for c in cat_col]).round(2)

    base_margin = {'Technology':0.18,'Furniture':0.08,'Office Supplies':0.22}
    discount = np.random.choice([0,0.1,0.2,0.3,0.4,0.5], n, p=[0.45,0.20,0.15,0.10,0.07,0.03])
    profit = np.array([sales[i] * (base_margin[cat_col[i]] - discount[i] * 0.8)
                       for i in range(n)]).round(2)

    dates = pd.date_range('2021-01-01','2024-12-31', periods=n)

    df = pd.DataFrame({
        'Order ID'   : [f'CA-{y}-{i:05d}' for i,y in enumerate(pd.DatetimeIndex(dates).year)],
        'Order Date' : dates,
        'Ship Mode'  : np.random.choice(ship_modes, n, p=[0.60,0.19,0.15,0.06]),
        'Segment'    : np.random.choice(segments, n, p=[0.52,0.30,0.18]),
        'Region'     : np.random.choice(regions, n, p=[0.32,0.28,0.22,0.18]),
        'Category'   : cat_col,
        'Sub-Category': subcat_col,
        'Sales'      : sales,
        'Quantity'   : np.random.randint(1,15,n),
        'Discount'   : discount,
        'Profit'     : profit,
    })
    print(f'✓ Sample data: {df.shape[0]:,} rows')

# Standardise columns
df.columns = df.columns.str.strip()
df['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce')
df['Year']  = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month
df['YearMonth'] = df['Order Date'].dt.to_period('M').astype(str)

df.head(3)

## 3. KPI Calculations

In [ ]:
total_sales   = df['Sales'].sum()
total_profit  = df['Profit'].sum()
profit_margin = total_profit / total_sales * 100
total_orders  = df['Order ID'].nunique()
avg_discount  = df['Discount'].mean() * 100
return_rate   = (df['Profit'] < 0).mean() * 100   # proxy: negative profit = returned/loss order

print(f'Total Sales    : ${total_sales:,.0f}')
print(f'Total Profit   : ${total_profit:,.0f}')
print(f'Profit Margin  : {profit_margin:.1f}%')
print(f'Total Orders   : {total_orders:,}')
print(f'Avg Discount   : {avg_discount:.1f}%')
print(f'Loss Order Rate: {return_rate:.1f}%')

## 4. Build Dashboard

In [ ]:
# ── Monthly sales trend ──────────────────────────────────────────────────────
monthly = (df.groupby('YearMonth')[['Sales','Profit']]
           .sum()
           .reset_index()
           .sort_values('YearMonth'))

# ── Region performance ───────────────────────────────────────────────────────
region = df.groupby('Region').agg(
    Sales=('Sales','sum'),
    Profit=('Profit','sum')
).reset_index()
region['Margin'] = region['Profit'] / region['Sales'] * 100

# ── Category breakdown ───────────────────────────────────────────────────────
cat_data = df.groupby('Category').agg(
    Sales=('Sales','sum'),
    Profit=('Profit','sum')
).reset_index()

# ── Sub-category profit ──────────────────────────────────────────────────────
subcat = (df.groupby('Sub-Category')['Profit']
          .sum()
          .sort_values()
          .reset_index())
subcat['Color'] = subcat['Profit'].apply(lambda x: DANGER if x < 0 else GREEN)

# ── Discount vs Profit scatter ───────────────────────────────────────────────
scatter_df = df.sample(min(2000, len(df)), random_state=42)

print('Data prepared ✓')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  BUILD THE FULL DASHBOARD
# ═══════════════════════════════════════════════════════════════════════════

fig = make_subplots(
    rows=4, cols=3,
    subplot_titles=(
        '','','',
        'Monthly Sales & Profit Trend','','',
        'Profit by Sub-Category','Revenue by Category','Region Performance',
        'Discount vs Profit Impact','Segment Mix','',
    ),
    specs=[
        [{'type':'indicator'},{'type':'indicator'},{'type':'indicator'}],
        [{'colspan':3},None,None],
        [{'type':'bar'},{'type':'pie'},{'type':'bar'}],
        [{'type':'scatter'},{'type':'pie'},{'type':'table'}],
    ],
    vertical_spacing=0.09,
    horizontal_spacing=0.06
)

# ── Row 1: KPI Indicators ────────────────────────────────────────────────────
fig.add_trace(go.Indicator(
    mode='number+delta',
    value=total_sales,
    title={'text':'Total Sales','font':{'size':14}},
    number={'prefix':'$','valueformat':',.0f','font':{'size':28,'color':ACCENT}},
), row=1, col=1)

fig.add_trace(go.Indicator(
    mode='number+gauge',
    value=profit_margin,
    title={'text':'Profit Margin','font':{'size':14}},
    number={'suffix':'%','valueformat':'.1f','font':{'size':28,'color':GREEN}},
    gauge={'axis':{'range':[0,40]},'bar':{'color':GREEN},'bgcolor':'#f0f0f0'},
), row=1, col=2)

fig.add_trace(go.Indicator(
    mode='number',
    value=total_orders,
    title={'text':'Total Orders','font':{'size':14}},
    number={'valueformat':',.0f','font':{'size':28,'color':'#2c3e50'}},
), row=1, col=3)

# ── Row 2: Monthly trend ─────────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=monthly['YearMonth'], y=monthly['Sales'],
    name='Sales', line=dict(color=ACCENT, width=2.5),
    fill='tozeroy', fillcolor='rgba(26,107,74,0.08)'
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=monthly['YearMonth'], y=monthly['Profit'],
    name='Profit', line=dict(color=LIGHT_G, width=2, dash='dot')
), row=2, col=1)

# ── Row 3: Sub-category profit bar ──────────────────────────────────────────
fig.add_trace(go.Bar(
    x=subcat['Profit'], y=subcat['Sub-Category'],
    orientation='h',
    marker_color=subcat['Color'],
    name='Profit by Sub-Cat',
    showlegend=False
), row=3, col=1)

# ── Row 3: Category pie ──────────────────────────────────────────────────────
fig.add_trace(go.Pie(
    labels=cat_data['Category'],
    values=cat_data['Sales'],
    hole=0.45,
    marker_colors=[ACCENT, GREEN, LIGHT_G],
    textinfo='label+percent',
    showlegend=False
), row=3, col=2)

# ── Row 3: Region bar ────────────────────────────────────────────────────────
fig.add_trace(go.Bar(
    x=region['Region'],
    y=region['Sales'],
    name='Sales by Region',
    marker_color=[ACCENT, GREEN, LIGHT_G, '#c8ede0'],
    showlegend=False
), row=3, col=3)

# ── Row 4: Discount vs Profit scatter ────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=scatter_df['Discount'],
    y=scatter_df['Profit'],
    mode='markers',
    marker=dict(
        color=scatter_df['Profit'],
        colorscale=[[0,DANGER],[0.5,'#f0f0f0'],[1,GREEN]],
        size=5, opacity=0.55, showscale=True,
        colorbar=dict(len=0.25, y=0.12, thickness=10)
    ),
    name='Discount vs Profit',
    showlegend=False
), row=4, col=1)

# ── Row 4: Segment pie ───────────────────────────────────────────────────────
seg = df.groupby('Segment')['Sales'].sum().reset_index()
fig.add_trace(go.Pie(
    labels=seg['Segment'],
    values=seg['Sales'],
    hole=0.45,
    marker_colors=[ACCENT, GREEN, LIGHT_G],
    showlegend=False
), row=4, col=2)

# ── Row 4: Region table ──────────────────────────────────────────────────────
fig.add_trace(go.Table(
    header=dict(
        values=['Region','Sales','Profit','Margin%'],
        fill_color=ACCENT, font_color='white', font_size=12,
        align='left'
    ),
    cells=dict(
        values=[
            region['Region'],
            region['Sales'].apply(lambda x: f'${x:,.0f}'),
            region['Profit'].apply(lambda x: f'${x:,.0f}'),
            region['Margin'].apply(lambda x: f'{x:.1f}%'),
        ],
        fill_color=[['#f8fdf9','#edf7f3']*4],
        font_size=12, align='left'
    )
), row=4, col=3)

# ── Layout ───────────────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text='<b>Retail Performance Dashboard</b><br><sup>Sales · Profit · Regional Analysis</sup>',
        x=0.01, font=dict(size=20, color='#1a1a18')
    ),
    height=1100,
    paper_bgcolor='#fafaf7',
    plot_bgcolor='#fafaf7',
    font=dict(family='Arial', size=11, color='#333'),
    legend=dict(orientation='h', y=1.01, x=0.4),
    margin=dict(t=100, b=40, l=40, r=40),
)

fig.update_xaxes(showgrid=True, gridcolor='#ebebeb', zeroline=False)
fig.update_yaxes(showgrid=True, gridcolor='#ebebeb', zeroline=False)

# ── Save interactive HTML ─────────────────────────────────────────────────────
out_path = '../retail_dashboard.html'
fig.write_html(out_path, include_plotlyjs='cdn')
print(f'✓ Dashboard saved → {out_path}')
print('  Open retail_dashboard.html in your browser to view it interactively!')

fig.show()

## 5. Key Insights

| # | Finding | Implication |
|---|---------|-------------|
| 1 | Technology has the highest sales but Furniture has thinner margins | Review furniture pricing strategy |
| 2 | Discounts above 30% consistently result in negative profit | Cap discounts at 20% max |
| 3 | Some sub-categories (e.g. Tables, Bookcases) are loss-making | Evaluate discontinuation or price revision |
| 4 | West region leads in both sales and profitability | Replicate West's playbook in Central |

## 6. Power BI Version

The same analysis was recreated in Power BI with DAX measures.  
See the `powerbi/` folder for the `.pbix` file (requires Power BI Desktop).